Import libraries and load dataset

In [1]:
import pandas as pd
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")

print("Dataset loaded successfully")
df.head()


Dataset loaded successfully


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,notify,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,notify,polite
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,notify,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,polite
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,polite


In [31]:
df.columns


Index(['id', 'sender', 'subject', 'body', 'priority', 'triage_label'], dtype='object')

In [32]:
df.shape


(200, 6)

In [39]:
if "ideal_intent" not in df.columns:
    df["ideal_intent"] = ""

if "ideal_tone" not in df.columns:
    df["ideal_tone"] = ""


In [41]:
def draft_ground_truth(text):
    text = str(text).lower()

    # Ideal Intent
    if any(word in text for word in [
        "urgent", "deadline", "overdue", "alert",
        "payment", "invoice", "security", "submit", "reminder"
    ]):
        intent = "notify"
    elif any(word in text for word in [
        "newsletter", "promotion", "sale", "offer",
        "unsubscribe", "marketing"
    ]):
        intent = "ignore"
    else:
        intent = "respond"

    # Ideal Tone
    if any(word in text for word in [
        "urgent", "alert", "overdue", "asap",
        "immediately", "security", "deadline"
    ]):
        tone = "urgent"
    elif any(word in text for word in [
        "please", "kindly", "thank you", "thanks"
    ]):
        tone = "polite"
    else:
        tone = "neutral"

    return intent, tone


In [42]:
df[["ideal_intent", "ideal_tone"]] = df["body"].apply(
    lambda x: pd.Series(draft_ground_truth(x))
)

df[["body", "ideal_intent", "ideal_tone"]].head()


,body,ideal_intent,ideal_tone
0,Reminder: The client meeting is scheduled at 1...,notify,neutral
1,Your invoice of INR 25515.09 is due on 2025-12...,notify,polite
2,Reminder: The client meeting is scheduled at 1...,notify,neutral
3,"Hello team, please find the attached weekly re...",respond,polite
4,"Hello team, please find the attached weekly re...",respond,polite


In [43]:
df.to_csv("../data/sample_emails_with_triage_200.csv", index=False)
print("Dataset updated with ground truth")


Dataset updated with ground truth


Email Assistant Logic

In [44]:
def email_assistant(email_text):
    text = str(email_text).lower()

    if any(word in text for word in [
        "urgent", "deadline", "alert", "overdue",
        "invoice", "payment", "security", "reminder"
    ]):
        return "notify", "urgent"

    elif any(word in text for word in [
        "thank you", "thanks", "appreciate"
    ]):
        return "ignore", "polite"

    else:
        return "respond", "neutral"


Generate Predictions

In [45]:
predictions = []

for idx, row in df.iterrows():
    intent, tone = email_assistant(row["body"])
    predictions.append({
        "id": idx,
        "predicted_intent": intent,
        "predicted_tone": tone
    })

pred_df = pd.DataFrame(predictions)
pred_df.head()


,id,predicted_intent,predicted_tone
0,0,notify,urgent
1,1,notify,urgent
2,2,notify,urgent
3,3,respond,neutral
4,4,respond,neutral


Merge Predictions with Ground Truth

In [46]:
eval_df = df.copy()
eval_df["predicted_intent"] = pred_df["predicted_intent"]
eval_df["predicted_tone"] = pred_df["predicted_tone"]

eval_df.head()


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone,predicted_intent,predicted_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,notify,neutral,notify,urgent
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,notify,polite,notify,urgent
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,notify,neutral,notify,urgent
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,polite,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,polite,respond,neutral


Evaluation Function

In [47]:
def evaluate(row):
    score = 0

    if row["predicted_intent"] == row["ideal_intent"]:
        score += 1

    if row["predicted_tone"] == row["ideal_tone"]:
        score += 1

    return score


Calculate accuracy

In [48]:
eval_df["score"] = eval_df.apply(evaluate, axis=1)

accuracy = (eval_df["score"].sum() / (len(eval_df) * 2)) * 100
accuracy


np.float64(78.0)

Error analysis

In [49]:
error_df = eval_df[eval_df["score"] < 2]

error_df[[
    "body",
    "ideal_intent",
    "predicted_intent",
    "ideal_tone",
    "predicted_tone"
]].head(20)

print("Total incorrect predictions:", error_df.shape[0])


Total incorrect predictions: 88


In [50]:
correct_df = eval_df[eval_df["score"] == 2]

correct_df[[
    "body",
    "ideal_intent",
    "predicted_intent",
    "ideal_tone",
    "predicted_tone"
]].head(20)

print("Total correct predictions:", correct_df.shape[0])


Total correct predictions: 112


In [51]:
eval_df.to_csv(
    "../data/milestone2_output_shwetha.csv",
    index=False
)

print("Milestone 2 output saved successfully")


Milestone 2 output saved successfully


<span style="font-size:28px; font-weight:bold">Which type of emails were hardest to classify?</span>


Emails with mixed intent or subtle wording were hardest to classify, as polite language could hide urgent meaning. Follow-ups or chained emails were also tricky because context from previous messages was needed.

<span style="font-size:28px; font-weight:bold">Why did your rules fail?</span>


Rule-based systems rely solely on keywords and cannot interpret context or implied meaning. They fail when emails contain multiple intents, ambiguous phrasing, or synonyms not in the rule set.

<span style="font-size:28px; font-weight:bold">How could an LLM improve this?</span>


LLMs understand context, tone, and semantics beyond simple keywords. They can handle mixed intent, implicit urgency, and varied phrasing more accurately than static rules.